<a href="https://colab.research.google.com/github/hamsini6/flipkartgrid/blob/main/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. ENVIRONMENT DEPENDENCIES & FILE UPLOAD
# ==========================================
!pip install pygeohash catboost lightgbm xgboost -q

import os
import pandas as pd
import numpy as np
import warnings
import pygeohash as pgh
from google.colab import files
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings('ignore')

# Prompt user to upload competition files if they are not already present
if not os.path.exists('train.csv') or not os.path.exists('test.csv'):
    print("Please upload train.csv and test.csv:")
    uploaded = files.upload()

# ==========================================
# 2. METRIC FUNCTION
# ==========================================
def calculate_r2_score(y_true, y_pred):
    """Calculates the specific competition metric: max(0, 100 * R2)."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / (ss_tot + 1e-9))
    return max(0, 100 * r2)

# ==========================================
# 3. FEATURE ENGINEERING ENGINE
# ==========================================
def engineer_features(train_path, test_path):
    print("[*] Loading raw datasets...")
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    y_train = train['demand'].values
    train_idx = train['Index']
    test_idx = test['Index']

    train = train.drop(columns=['demand'])
    df = pd.concat([train, test], axis=0).reset_index(drop=True)

    print("[*] Processing missing states...")
    df['RoadType'] = df['RoadType'].fillna('Missing_Road')
    df['Weather'] = df['Weather'].fillna('Missing_Weather')

    # Spatial-temporal median imputation for Temperature
    df['Temperature'] = df.groupby(['day', 'timestamp'])['Temperature'].transform(lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

    print("[*] Decoding Geohashes into Exact Coordinates...")
    # High-fidelity decoding via pygeohash
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else 0.0)
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else 0.0)
    df['geohash_prefix5'] = df['geohash'].astype(str).str[:5]

    print("[*] Engineering Cyclical Temporal features...")
    time_split = df['timestamp'].str.split(':', expand=True)
    df['hour'] = time_split[0].astype(int)
    df['minute'] = time_split[1].astype(int)
    df['total_minutes'] = df['hour'] * 60 + df['minute']

    # Cyclical transformations mapping 23:59 close to 00:00
    df['sin_time'] = np.sin(2 * np.pi * df['total_minutes'] / 1440.0)
    df['cos_time'] = np.cos(2 * np.pi * df['total_minutes'] / 1440.0)

    # Domain-specific rush hour flag
    df['is_rush_hour'] = (((df['hour'] >= 8) & (df['hour'] <= 10)) | ((df['hour'] >= 17) & (df['hour'] <= 19))).astype(int)

    print("[*] Generating cross-feature interactions...")
    df['lane_vehicle_interact'] = df['NumberofLanes'].astype(str) + "_" + df['LargeVehicles'].astype(str)
    df['temp_weather_interact'] = df['Weather'] + "_" + df['Temperature'].round(-1).astype(str)

    # High-cardinality Frequency Encoding
    for col in ['geohash', 'geohash_prefix5']:
        freq = df[col].value_counts().to_dict()
        df[f'{col}_freq'] = df[col].map(freq)

    # Categorical Label Encoding
    cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
                'geohash', 'geohash_prefix5', 'lane_vehicle_interact', 'temp_weather_interact']

    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    X_train = df.iloc[:len(train)].copy()
    X_test = df.iloc[len(train):].copy()

    return X_train, X_test, y_train, train_idx, test_idx

# Run transformation matrix pipeline
X_train, X_test, y_train, train_idx, test_idx = engineer_features('train.csv', 'test.csv')

features_to_drop = ['Index', 'timestamp']
X_train = X_train.drop(columns=features_to_drop)
X_test = X_test.drop(columns=features_to_drop)

# ==========================================
# 4. LEAKAGE-SAFE CROSS-VALIDATION ENGINE
# ==========================================
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

X_train['target_enc_geohash'] = 0.0
X_test['target_enc_geohash'] = 0.0
global_mean = y_train.mean()

print("\n[*] Initializing Out-of-Fold Cross Validation Loop...")
for fold, (train_idx_cv, val_idx_cv) in enumerate(kf.split(X_train, y_train)):
    print(f"--- Processing Fold {fold + 1} / {FOLDS} ---")

    # Internal target encoding calculation to enforce a strict validation wall
    fold_train_geo = X_train.iloc[train_idx_cv].copy()
    fold_train_geo['demand'] = y_train[train_idx_cv]
    geo_means = fold_train_geo.groupby('geohash')['demand'].mean()

    X_train.loc[X_train.index[val_idx_cv], 'target_enc_geohash'] = X_train.iloc[val_idx_cv]['geohash'].map(geo_means).fillna(global_mean)

    X_tr, y_tr = X_train.iloc[train_idx_cv], y_train[train_idx_cv]
    X_va, y_va = X_train.iloc[val_idx_cv], y_train[val_idx_cv]

    # Model 1: LightGBM Regressor
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.04, num_leaves=63,
        random_state=42, subsample=0.8, colsample_bytree=0.8, n_jobs=-1
    )
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx_cv] = model_lgb.predict(X_va)

    # Model 2: XGBoost Regressor (Optimized to detect GPU environments in Colab)
    model_xgb = xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.04, max_depth=6,
        random_state=42, subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', device='cuda' if xgb.__version__ >= '2.0.0' else 'gpu_hist', n_jobs=-1
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx_cv] = model_xgb.predict(X_va)

    # Model 3: CatBoost Regressor
    model_cat = cb.CatBoostRegressor(
        iterations=1500, learning_rate=0.05, depth=6,
        random_state=42, verbose=False, early_stopping_rounds=50
    )
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[val_idx_cv] = model_cat.predict(X_va)

# Apply global target mapping to testing feature space
final_geo_means = pd.DataFrame({'geohash': X_train['geohash'], 'demand': y_train}).groupby('geohash')['demand'].mean()
X_test['target_enc_geohash'] = X_test['geohash'].map(final_geo_means).fillna(global_mean)

# ==========================================
# 5. METRIC RESULTS & EVALUATION REPORT
# ==========================================
score_lgb = calculate_r2_score(y_train, oof_lgb)
score_xgb = calculate_r2_score(y_train, oof_xgb)
score_cat = calculate_r2_score(y_train, oof_cat)

print("\n" + "="*45)
print("       COLAB METRIC SCORES ENGINE      ")
print("="*45)
print(f" LightGBM Regressor Score (100*R²) : {score_lgb:.4f}")
print(f" XGBoost Regressor Score (100*R²)  : {score_xgb:.4f}")
print(f" CatBoost Regressor Score (100*R²) : {score_cat:.4f}")
print("="*45)

# ==========================================
# 6. ENSEMBLE TRAINING & PREDICTION GENERATION
# ==========================================
print("\n[*] Fitting meta-ensemble on full training population...")
final_lgb = lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.04, num_leaves=63, random_state=42, n_jobs=-1)
final_xgb = xgb.XGBRegressor(n_estimators=1200, learning_rate=0.04, max_depth=6, random_state=42, tree_method='hist', n_jobs=-1)
final_cat = cb.CatBoostRegressor(iterations=1200, learning_rate=0.05, depth=6, random_state=42, verbose=False)

final_lgb.fit(X_train, y_train)
final_xgb.fit(X_train, y_train)
final_cat.fit(X_train, y_train)

preds_lgb = final_lgb.predict(X_test)
preds_xgb = final_xgb.predict(X_test)
preds_cat = final_cat.predict(X_test)

# Blending using variance weight distribution
final_predictions = (0.40 * preds_lgb) + (0.35 * preds_xgb) + (0.25 * preds_cat)

# Create final submission file
submission = pd.DataFrame({
    'Index': test_idx,
    'demand': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("\n[✓] Finished successfully! 'submission.csv' is saved and ready in your files side panel.")

Please upload train.csv and test.csv:


Saving test.csv to test.csv
Saving train.csv to train (1).csv
[*] Loading raw datasets...
[*] Processing missing states...
[*] Decoding Geohashes into Exact Coordinates...
[*] Engineering Cyclical Temporal features...
[*] Generating cross-feature interactions...

[*] Initializing Out-of-Fold Cross Validation Loop...
--- Processing Fold 1 / 5 ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014249 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1180
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 21
[LightGBM] [Info] Start training from score 0.093784
--- Processing Fold 2 / 5 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005963 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1434
[LightGBM] [Info] Nu